# HSTU_CANONICAL_v1 — finalizer (checkpoint auto-discovery patch)

This wrapper executes the canonical finalizer from the repo, then overrides only
checkpoint resolution so it can find `best.pt`, `latest.pt`, or `last.pt`
anywhere under `MyDrive`.

It does **not** retrain or change HSTU weights/quality logic.

In [ ]:
import os, json, urllib.request
from pathlib import Path
import torch

assert torch.cuda.is_available(), "Switch Colab runtime to GPU."

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

try:
    import triton
    print("Triton:", triton.__version__)
except Exception:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "triton"], check=True)
    import triton
    print("Triton installed:", triton.__version__)

In [ ]:
# Load the canonical finalizer notebook from GitHub and execute definition cells.
RAW = "https://raw.githubusercontent.com/hanialshater/Sparsewalker-/main/experiments/hstu_reproduction/HSTU_CANONICAL_v1_Finalize_Colab.ipynb"
raw_nb = json.loads(urllib.request.urlopen(RAW).read().decode("utf-8"))

skip_markers = (
    'resolve_checkpoint("core")',
    "stamp, summary_df, latency_df = finalize()",
    'print("HSTU_CANONICAL_v1:"',
)

executed = 0
for cell in raw_nb["cells"]:
    if cell.get("cell_type") != "code":
        continue
    src = "".join(cell.get("source", []))
    if any(m in src for m in skip_markers):
        continue
    if src.strip():
        exec(compile(src, "<canonical-finalizer-cell>", "exec"), globals(), globals())
        executed += 1

print("Loaded canonical finalizer definition cells:", executed)

In [ ]:
# ------------------------------------------------------------------
# PATCH: robust checkpoint discovery across all of MyDrive.
# ------------------------------------------------------------------

CHECKPOINT_OVERRIDES = {"core": None, "large": None}
_RESOLVED_CHECKPOINTS = {}

def _checkpoint_ndcg(ck):
    metrics = ck.get("metrics", {}) or {}
    if "NDCG@10" in metrics:
        try:
            return float(metrics["NDCG@10"])
        except Exception:
            pass
    hist = ck.get("history", []) or []
    vals = []
    for row in hist:
        if isinstance(row, dict) and row.get("NDCG@10") is not None:
            vals.append(float(row["NDCG@10"]))
        elif isinstance(row, dict) and row.get("val_NDCG@10") is not None:
            vals.append(float(row["val_NDCG@10"]))
    return max(vals) if vals else float("-inf")

def _checkpoint_variant(ck, path):
    cfg = ck.get("config", {}) or {}
    v = cfg.get("model")
    if v in ("core", "large"):
        return v
    name = str(path).lower()
    if "hstu_core" in name or "/core" in name:
        return "core"
    if "hstu_large" in name or "/large" in name:
        return "large"
    return None

def resolve_checkpoint(variant):
    if variant in _RESOLVED_CHECKPOINTS:
        return _RESOLVED_CHECKPOINTS[variant]

    override = CHECKPOINT_OVERRIDES.get(variant)
    if override:
        p = Path(override)
        if not p.exists():
            raise FileNotFoundError(f"Override does not exist: {p}")
        _RESOLVED_CHECKPOINTS[variant] = p
        return p

    mydrive = Path("/content/drive/MyDrive")
    expected_dir = mydrive / "hstu_pure_pytorch_ml1m" / f"hstu_{variant}_seed42"

    candidates = []
    for fname in ("best.pt", "latest.pt", "last.pt"):
        p = expected_dir / fname
        if p.exists():
            candidates.append(p)

    for fname in ("best.pt", "latest.pt", "last.pt"):
        pattern = f"**/hstu_{variant}_seed42/{fname}"
        for p in mydrive.glob(pattern):
            if p not in candidates:
                candidates.append(p)

    if not candidates:
        for p in mydrive.glob("**/*.pt"):
            low = str(p).lower()
            if "hstu" in low and variant in low:
                candidates.append(p)

    inspected = []
    for p in candidates:
        try:
            ck = torch.load(p, map_location="cpu", weights_only=False)
            found_variant = _checkpoint_variant(ck, p)
            ndcg = _checkpoint_ndcg(ck)
            epoch = int(ck.get("epoch", -1))
            inspected.append((p, found_variant, ndcg, epoch))
        except Exception as e:
            print("Skipping unreadable checkpoint:", p, repr(e))

    matches = [r for r in inspected if r[1] == variant]
    if not matches:
        print("\nCould not resolve", variant)
        print("Expected directory:", expected_dir)
        print("Candidates inspected:")
        for row in inspected:
            print(" ", row)
        print("\nHSTU-like Drive paths:")
        for p in list(mydrive.glob("**/*hstu*"))[:200]:
            print(" ", p)
        raise FileNotFoundError(
            f"No {variant} checkpoint found. "
            f"If needed, set CHECKPOINT_OVERRIDES['{variant}'] to the exact .pt path."
        )

    def score(row):
        p, _, ndcg, epoch = row
        bonus = 2 if p.name == "best.pt" else (1 if p.name == "latest.pt" else 0)
        return (ndcg, bonus, epoch)

    chosen = max(matches, key=score)
    p, _, ndcg, epoch = chosen
    print("RESOLVED CHECKPOINT:", {
        "variant": variant,
        "path": str(p),
        "epoch": epoch,
        "saved_NDCG@10": ndcg,
    })
    _RESOLVED_CHECKPOINTS[variant] = p
    return p

def checkpoint_path(variant):
    return resolve_checkpoint(variant)

def load_canonical_model(variant, max_item_id):
    path = resolve_checkpoint(variant)
    ck = torch.load(path, map_location=DEVICE, weights_only=False)

    cfg = ck["config"]
    n = int(cfg["max_len"] + cfg["max_output_len"])
    g = geometry(variant)

    model = HSTU(
        max_item_id=max_item_id,
        n=n,
        dropout=float(cfg["dropout"]),
        l2_eps=float(cfg.get("l2_eps", 1e-6)),
        **g,
    ).to(DEVICE)
    model.load_state_dict(ck["model"])
    model.eval()

    print("CHECKPOINT", variant, path, "epoch", ck.get("epoch"), ck.get("metrics", {}))
    return model, ck

print("Core:", resolve_checkpoint("core"))
print("Large:", resolve_checkpoint("large"))

In [ ]:
# Run the canonical stamp suite with the patched checkpoint resolver.
stamp, summary_df, latency_df = finalize()
display(summary_df)
display(latency_df)

if len(latency_df):
    display(
        latency_df.pivot_table(
            index=["variant", "batch", "history"],
            columns="backend",
            values="p50_ms",
        ).reset_index()
    )

print("HSTU_CANONICAL_v1:", stamp["status"])
print("Stamp file:", "/content/drive/MyDrive/hstu_pure_pytorch_ml1m/HSTU_CANONICAL_v1_stamp.json")